In [5]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [6]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater, VaspBuilderUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm
from matchest.aiida_utils.pmg import load_mp_struct
from ase.symbols import Formula
from aiida.engine import submit

from matchest.aiida_utils.workflows.simple_vac import SimpleVacancyWorkChain

In [7]:
basepath = GroupPathX('hc-defect')
workpath = basepath['workflows']
elemental_struct_path = GroupPathX('defects/elemental_ref')

In [8]:
elemental_struct_path.show_tree()

elemental_ref
├── As_bulk *
├── Au_bulk *
├── Ba_bulk *
├── Bi_bulk *
├── Ca_bulk *
├── Cd_bulk *
├── Cu_bulk *
├── Ge_bulk *
├── Hg_bulk *
├── In_bulk *
├── Mg_bulk *
├── O_bulk *
├── P_bulk *
├── Pb_bulk *
├── S_bulk *
├── Sb_bulk *
├── Se_bulk *
├── Sn_bulk *
├── Sr_bulk *
├── Te_bulk *
├── Tl_bulk *
└── Zn_bulk *



## Define the input to the wokchain

Generate structure

In [9]:
form = Formula('InCuSe2')  # Change the formula here
symbols = list(form)
ASite = symbols[0]
BSite = symbols[1]
CSite = symbols[2]

structure = load_mp_struct('mp-14090')

ps = structure.get_pymatgen()
ps['Tl'] = 'He'
ps['Cu'] = 'Ne' 
ps['Se'] = 'Xe'

ps['He'] = ASite  
ps['Ne'] = BSite 
ps['Xe'] = CSite

structure = orm.StructureData(pymatgen=ps)
print(ps)

Full Formula (In2 Cu2 Se4)
Reduced Formula: InCuSe2
abc   :   5.887256   5.887381   7.244696
angles: 113.984386 113.971472  89.992190
pbc   :       True       True       True
Sites (8)
  #  SP            a         b          c
---  ----  ---------  --------  ---------
  0  In     0.499996  0.500005   3.3e-05
  1  In     0.750005  0.249948   0.500033
  2  Cu    -0.000138  0.00018   -0.000157
  3  Cu     0.250046  0.749934   0.499851
  4  Se     0.32851   0.374992   0.250027
  5  Se     0.921433  0.874905   0.249837
  6  Se     0.124984  0.671439   0.750068
  7  Se     0.625163  0.078597   0.750309


In [10]:
builder = SimpleVacancyWorkChain.get_builder()

upd = VaspRelaxUpdater(builder = builder.relax, ).apply_preset(structure, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1, 'ncore':8 , 'kpar': 4, 'isym': 0, 'symprec': 1e-9}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600*12, queue_name='xhhctdnormal')
upd.set_label(f'{form} RELAX')
upd.set_relax_settings(algo='rd')
# Assign the elemental structures
builder.elemental_structures = {symbols[i]: elemental_struct_path[symbols[i] + '_bulk'].node 
                                for i in range(3)}
assert None not in builder.elemental_structures.values()
builder.supercell_dim = orm.List([2,2,2])  # 222 supercell
# Updated parameters for supercell calculation
builder.supercell_workchain_updates = orm.Dict(
    {'options': {
        'resources': {'tot_num_mpiprocs': 128, 'num_machines':2},
        'max_wallclock_seconds': 3600 * 12, 
        'queue_name': 'xhhctdnormal',
        },
     'incar': {'kpar': 2, 'ncore': 8, 'isym': 0, 'symprec': 1e-9}
    }
)

In [11]:
workpath.show_tree(decorate_by=['exit_status', 'pk'])

workflows 121
├── Ba3PbO_work [0] | 668617
├── Ba3SnO_work [0] | 668797
├── Ca2Pb_work [0] | 669135
├── Ca2Sn_work [0] | 669224
├── Ca3Bi2_work [0] | 669702
├── Ca3PbO_work [0] | 668910
├── GeCdAs2_work [0] | 664271
├── GeCdSb2_work [0] | 664179
├── GeZnBi2_work [0] | 664223
├── GeZnSb2_work [excepted] | 664317
├── GeZnSb2_work_sym_update [0] | 665530
├── InAuSe2_work_sym_update [0] | 666598
├── InCuSe2_work_sym_update [0] | 670540
├── Mg2Pb_work [501] | 670744
├── Mg2Sn_work [501] | 670766
├── Mg3Bi2_work [0] | 669492
├── Mg3PbO_work [0] | 668955
├── PbCdAs2_work [excepted] | 664341
├── PbCdAs2_work_sym_update [0] | 665552
├── PbCdP2_work [excepted] | 664132
├── PbCdP2_work_sym_update [0] | 665574
├── PbZnAs2_work [excepted] | 664157
├── PbZnAs2_work_sym_update [0] | 665596
├── PbZnP2_work [excepted] | 664249
├── PbZnP2_work_sym_update [0] | 665618
├── PbZnSb2_work [0] | 663926
├── SnCdSb2_work [0] | 664110
├── SnZnBi2_work [0] | 664201
├── SnZnSb2_work [excepted] | 664293
├── SnZnSb2

In [12]:
from collections import defaultdict

In [13]:
results = defaultdict(lambda : {})
for path in workpath:
    node = path.node
    if node.is_finished_ok:
        for kind, eng in node.outputs.vacancy_formation_energies.items():
            ref_structure = node.inputs.relax.structure.get_ase()
            form = ref_structure.symbols.formula.reduce()[0]
            #print(ref_structure.symbols.formula.reduce()[0], kind, eng[kind])
            results[str(form)][kind] = eng[kind]

In [14]:
import matplotlib.pyplot as plt

In [15]:
for key,value in results.items():
    print(f"{key}  max: {max(value.values()):.3f} eV min: {min(value.values()):.3f} eV")

PbZnSb2  max: 1.771 eV min: 0.715 eV
SnCdSb2  max: 2.028 eV min: 1.242 eV
GeCdSb2  max: 1.427 eV min: 1.115 eV
SnZnBi2  max: 1.277 eV min: 0.278 eV
GeZnBi2  max: 1.457 eV min: 0.228 eV
GeCdAs2  max: 2.363 eV min: 1.786 eV
GeZnSb2  max: 1.586 eV min: 0.972 eV
PbCdAs2  max: 1.712 eV min: 0.548 eV
PbCdP2  max: 2.520 eV min: 0.418 eV
PbZnAs2  max: 2.683 eV min: 0.759 eV
PbZnP2  max: 3.494 eV min: 0.606 eV
SnZnSb2  max: 2.147 eV min: 1.101 eV
InAuSe2  max: 1.885 eV min: -0.284 eV
TlCuSe2  max: 1.600 eV min: 0.581 eV
Ba3PbO  max: 6.012 eV min: 1.538 eV
Ba3SnO  max: 6.053 eV min: 1.640 eV
Ca3PbO  max: 6.601 eV min: 1.893 eV
Mg3PbO  max: 4.812 eV min: 0.225 eV
Sr3PbO  max: 6.496 eV min: 1.557 eV
Sr3SnO  max: 6.576 eV min: 1.745 eV
Ca2Pb  max: 2.636 eV min: 1.604 eV
Ca2Sn  max: 2.897 eV min: 1.843 eV
Sr2Pb  max: 2.483 eV min: 2.045 eV
Mg3Bi2  max: 1.802 eV min: 1.166 eV
Ca3Bi2  max: 3.112 eV min: 2.673 eV
Sr3Bi2  max: 3.049 eV min: 2.823 eV
InCuSe2  max: 2.370 eV min: 0.419 eV


In [16]:
for key,value in results.items():
    print(f"{key}  {value}")
    

PbZnSb2  {'V_Sb': 0.71479827499999, 'V_Zn': 1.087917885, 'V_Pb': 1.77083034}
SnCdSb2  {'V_Sn': 2.02804724, 'V_Sb': 1.322993695, 'V_Cd': 1.24150532}
GeCdSb2  {'V_Cd': 1.11549563, 'V_Sb': 1.427258665, 'V_Ge': 1.258317645}
SnZnBi2  {'V_Bi': 1.27651988, 'V_Sn': 1.23147821, 'V_Zn': 0.27801612499999}
GeZnBi2  {'V_Bi': 1.45665572, 'V_Zn': 0.22768930499998, 'V_Ge': 0.63207366499998}
GeCdAs2  {'V_Cd': 1.83761919, 'V_As': 1.786082485, 'V_Ge': 2.363226465}
GeZnSb2  {'V_Sb': 1.586109385, 'V_Ge': 1.48991149, 'V_Zn': 0.97153344499998}
PbCdAs2  {'V_Pb': 1.69315752, 'V_As': 0.54839565, 'V_Cd': 1.711862785}
PbCdP2  {'V_P': 0.41840695249999, 'V_Pb': 0.88482364999999, 'V_Cd': 2.519696975}
PbZnAs2  {'V_Pb': 2.68301708, 'V_As': 0.7593013, 'V_Zn': 1.800187285}
PbZnP2  {'V_P': 0.60580937249995, 'V_Pb': 3.49410607, 'V_Zn': 2.657185675}
SnZnSb2  {'V_Sb': 1.456566975, 'V_Sn': 2.14746784, 'V_Zn': 1.100572375}
InAuSe2  {'V_Au': -0.28426346000001, 'V_Se': 1.53290017, 'V_In': 1.88490381}
TlCuSe2  {'V_Cu': 0.5811214

## Print minimum and maximum defect formation energies

In [17]:
cases="""InAuSe2
TlCuSe2
PbZnSb2
PbZnSb2
GeCdSb2
SnCdSb2
PbCdP2
PbCdAs2
PbZnAs2
GeCdSb2"""
for line in cases.split('\n'):
    if line in results:
        print(f"{min(results[line].values()):.3f}")
    else:
        print('NA')

-0.284
0.581
0.715
0.715
1.115
1.242
0.418
0.548
0.759
1.115


In [61]:
cases="""InCuSe2
InCuSe2
SnZnSb2
GeZnSb2
GeZnSb2
SnZnSb2
PbZnP2
GeCdAs2
PbZnP2
GeCdAs2"""
for line in cases.split('\n'):
    if line in results:
        print(f"{min(results[line].values()):.3f}")
    else:
        print('MISSING')

MISSING
MISSING
1.101
0.972
0.972
1.101
0.606
1.786
0.606
1.786


In [58]:
cases="""InAuSe2
TlCuSe2
PbZnSb2
PbZnSb2
GeCdSb2
SnCdSb2
PbCdP2
PbCdAs2
PbZnAs2
GeCdSb2"""
for line in cases.split('\n'):
    if line in results:
        print(f"{max(results[line].values()):.3f}")
    else:
        print('NA')

1.885
1.600
1.771
1.771
1.427
2.028
2.520
1.712
2.683
1.427


In [60]:
cases="""InCuSe2
InCuSe2
SnZnSb2
GeZnSb2
GeZnSb2
SnZnSb2
PbZnP2
GeCdAs2
PbZnP2
GeCdAs2"""
for line in cases.split('\n'):
    if line in results:
        print(f"{max(results[line].values()):.3f}")
    else:
        print('MISSING')

MISSING
MISSING
2.147
1.586
1.586
2.147
3.494
2.363
3.494
2.363
